## Notebook to run code

### Run file by file

In [ ]:
inputDir = 'input'
outputDir = 'output'
regulationDir = 'WHS procedure'
filename = 'csiro_Electrical' # without .pdf

In [47]:
!python -m main --pdf "$inputDir"/"$regulationDir"/"$filename".pdf \
                --out "$outputDir"/"$regulationDir"/"$filename".json

OCR mode: False
[Parallel(n_jobs=128)]: Using backend LokyBackend with 128 concurrent workers.
[Parallel(n_jobs=128)]: Done   2 out of  26 | elapsed:    0.5s remaining:    6.5s
[Parallel(n_jobs=128)]: Done   5 out of  26 | elapsed:    0.6s remaining:    2.3s
[Parallel(n_jobs=128)]: Done   8 out of  26 | elapsed:    0.6s remaining:    1.3s
[Parallel(n_jobs=128)]: Done  11 out of  26 | elapsed:    0.6s remaining:    0.8s
[Parallel(n_jobs=128)]: Done  14 out of  26 | elapsed:    0.6s remaining:    0.5s
[Parallel(n_jobs=128)]: Done  17 out of  26 | elapsed:    0.6s remaining:    0.3s
[Parallel(n_jobs=128)]: Done  20 out of  26 | elapsed:    0.6s remaining:    0.2s
[Parallel(n_jobs=128)]: Done  23 out of  26 | elapsed:    0.6s remaining:    0.1s
[Parallel(n_jobs=128)]: Done  26 out of  26 | elapsed:    0.7s finished
Obtaining page positions...
Title: 100%|█████████████████████████████████| 838/838 [00:06<00:00, 120.39it/s]
831 warnings logged to temp_mineru/model WHS regulations/auto/warnin

### Run all at once

In [ ]:
import os

inputDir = 'input'
outputDir = 'output'
regulationDirs = os.listdir(os.path.join(inputDir))
for regulationDir in regulationDirs:
    filenames = [os.path.splitext(f)[0] for f in os.listdir(os.path.join(inputDir, regulationDir)) if f.endswith('.pdf')] # remove .pdf
    for filename in filenames:
        inputPath = f'{inputDir}/{regulationDir}/{filename}.pdf'
        outputPath = f'{outputDir}/{regulationDir}/{filename}.json'
        if not os.path.exists(outputPath): # run if output file does not exist
            os.system(f"python main.py --pdf \"{inputDir}/{regulationDir}/{filename}.pdf\" \
                                       --out \"{outputDir}/{regulationDir}/{filename}.json\"")

## Update page numbers

In [ ]:
import json
from copy import deepcopy
from helper import obtain_positions, CompactPositionEncoder
from pylatexenc.latex2text import LatexNodes2Text

def _f(text: str) -> str:
    """ Remove formulas present in text. """
    if "$\\" in text:
        return LatexNodes2Text().latex_to_text(text)
    return text

def flatten_json(json_data):
    '''
    Flatten the nested JSON structure into a list of entries.
    '''
    flat_list = []

    def _flatten(entries):
        for entry in entries:
            _entry = deepcopy(entry)
            # Remove formulas
            _entry['title'] = _f(_entry['title'])
            _entry['text'] = _f(_entry['text'])
            _entry.pop('subsegments', None)
            flat_list.append(_entry)
            if 'subsegments' in entry and isinstance(entry['subsegments'], list):
                _flatten(entry['subsegments'])

    _flatten(json_data)
    return flat_list

# Update the original JSON with positions
def update_nested(json_data, flat_data):
    duplicate_ids = [item['id'] for item in flat_data if flat_data.count(item) > 1]
    if duplicate_ids: raise ValueError(f"Duplicate IDs found in flat_data: {duplicate_ids}")

    flat_index = 0

    def _update(entries):
        nonlocal flat_index
        for entry in entries:
            if flat_index < len(flat_data) and entry['id'] == flat_data[flat_index]['id']:
                entry['position'] = flat_data[flat_index]['position']
                entry['page'] = flat_data[flat_index]['page']
                entry['title'] = flat_data[flat_index]['title']
                entry['text'] = flat_data[flat_index]['text']
                flat_index += 1
            if 'subsegments' in entry and isinstance(entry['subsegments'], list):
                _update(entry['subsegments'])

    _update(json_data)
    return json_data

In [3]:
directory = 'Australia WHS regulation'
jurisdiction = 'NSW - Work Health and Safety Regulation 2017'
input_pdf = f'input/{directory}/{jurisdiction}.pdf'
input_json = f'output/{directory}/{jurisdiction}.json'

with open(input_json, 'r') as f:
    data = json.load(f)
    flat_data = flatten_json(data['content'])

flat_data = obtain_positions(input_pdf, flat_data, ocr=False, similarity_threshold=0.94, skip_toc=15)
data['content'] = update_nested(data['content'], flat_data)
with open(input_json, 'w') as f:
    json.dump(data, f, indent=4, ensure_ascii=True, cls=CompactPositionEncoder)

Title: 100%|██████████| 885/885 [00:02<00:00, 414.41it/s] 

2 warnings logged to temp_mineru/NSW - Work Health and Safety Regulation 2017/auto/warnings.log


In [7]:
mic_err = f'{43/1065*100:.2f}%'
florida_err = f'{14/725*100:.2f}%'
texas_err = f'{36/510*100:.2f}%'

print(f'Michigan: {mic_err} error'
      f'\nFlorida: {florida_err} error'
      f'\nTexas: {texas_err} error')

Michigan: 4.04% error
Florida: 1.93% error
Texas: 7.06% error


### Detect Changes

In [8]:
!pip install GitPython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.6/208.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 41.2 MB/s eta 0:00:00


In [16]:
from git import Repo, GitCommandError

def get_file_at_commit_gitpython(repo_path: str, commit: str, file_path: str) -> str:
    repo = Repo(repo_path)
    try:
        data: str = repo.git.show(f"{commit}:{file_path}", with_extended_output=False, as_process=False, decode=False)
        return data
    except GitCommandError as e:
        raise FileNotFoundError(str(e)) from e

In [ ]:
import json

directory = 'human study/chunking/flattened'
result = dict()
for filename in ['Florida Uniform Traffic Control Law.json', 'Michigan Vehicle Code.json', 'Texas Rules of the Road.json']:
    content = get_file_at_commit_gitpython(repo_path='..', commit='HEAD~1', file_path=f'{directory}/{filename}')
    data_prev = json.loads(content)

    with open(f'../{directory}/{filename}', 'r') as f:
        data_current = json.load(f)

    prev_by_id = {item['id']: item for item in data_prev}
    curr_by_id = {item['id']: item for item in data_current}

    # Items to be removed from the previous version
    changed_items = {'added': [], 'updated': [], 'removed': []}
    for entry_id, prev_entry in prev_by_id.items():
        # check title, text is the same
        if entry_id in curr_by_id:
            curr_entry = curr_by_id[entry_id]
            if prev_entry['title'] != curr_entry['title'] or prev_entry['text'] != curr_entry['text']:
                changed_items['updated'].append(curr_entry)
        else:
            changed_items['removed'].append(prev_entry)

    # Items to be added
    for entry_id, curr_entry in curr_by_id.items():
        if entry_id not in prev_by_id:
            changed_items['added'].append(curr_entry)

    result[filename] = changed_items

    print(f'For {filename}:')
    print(f'Added items: {len(changed_items["added"])}')
    print(f'Updated items: {len(changed_items["updated"])}')
    print(f'Removed items: {len(changed_items["removed"])}')

# Save
with open('test/us_changed_items.json', 'w') as f:
    json.dump(result, f, indent=4)

For Florida Uniform Traffic Control Law.json:
Added items: 0
Updated items: 31
Removed items: 0
For Michigan Vehicle Code.json:
Added items: 13
Updated items: 98
Removed items: 27
For Texas Rules of the Road.json:
Added items: 0
Updated items: 463
Removed items: 3


In [20]:
print(type(content))

<class 'str'>


### Update incomplete titles
* Occurs in Michigan provisions where `Sec. n` detected but the full title is not parsed.
* Gemini detects the full title, so we can update the JSON with the correct titles.

In [5]:
prompt = 'I want you to extract section names from given pdf for below list of sections and return heading as a python list of strings. Do not rephrase or edit section name. Keep it in original format. eg: "257.1c “Articulated bus” defined."'
incomplete_titles = [(index, item['title']) for index, item in enumerate(flat_data) if item['title'].startswith('Sec.')]
for i, title in incomplete_titles:
    prompt += f"\n{i+1}. {title}"

In [ ]:
# As detected by Gemini
# sections = []

# Update flat_data with the detected sections
for i, (index, _) in enumerate(incomplete_titles):
    flat_data[index]['title'] = sections[i]
data['content'] = update_nested(data['content'], flat_data)
with open(input_json, 'w') as f:
    json.dump(data, f, indent=4)
